# DIVRS — Train PROD trên Colab (ML-10M)

Code: **[HatDuaa/DIVRS-reproduce](https://github.com/HatDuaa/DIVRS-reproduce)**. Train ghi **checkpoint mỗi epoch thẳng vào Drive/Colab Notebooks** → ngắt phiên vẫn còn.

⚙️ Bật **T4 GPU**: Runtime → Change runtime type → T4 GPU.

## 1. GPU + lib

In [ ]:
!nvidia-smi || echo 'KHONG co GPU -> chay CPU (cham)'
!nproc
!pip install -q absl-py setproctitle deprecated faiss-cpu
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Lấy code (hard reset về bản mới nhất GitHub)

In [ ]:
import os
REPO = '/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
!git log --oneline -1

## 3. Tải data ML-10M

In [ ]:
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
need = ['train_coo_record.npz','val_coo_record.npz','test_coo_record.npz',
        'train_skew_coo_record.npz','popularity.npy','popularity_blend.npy']
miss = [f for f in need if not os.path.exists('data/ml10m/output/'+f)]
print('Data thieu:', miss if miss else 'KHONG — du, san sang!')

## 4. Mount Drive → thư mục lưu kết quả
Train sẽ ghi thẳng vào đây (weight + log) **mỗi epoch**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/DIVRS_output'
os.makedirs(OUT_DIR, exist_ok=True)
print('Luu ket qua vao:', OUT_DIR)

## 5. Train PROD + Test
`--output` trỏ thẳng Drive → **mỗi epoch ghi `ckpt/epoch_N.pth` lên Drive** (ngắt phiên vẫn còn). `epochs=50` cap, `es_patience=5` tự dừng khi val recall bão hòa.

In [ ]:
import torch
USE_GPU = torch.cuda.is_available()
print('use_gpu =', USE_GPU)
!python app.py \
  --flagfile config/ml10m_DIVRS.cfg \
  --output "{OUT_DIR}/" \
  --use_gpu={USE_GPU} --gpu_id=0 \
  --cg_use_gpu=False \
  --num_workers=2 \
  --epochs=50 --es_patience=5

## 6. Xem kết quả (Recall@K / NDCG) — đọc thẳng từ Drive

In [ ]:
import glob
run = max(glob.glob(OUT_DIR+'/*/'), key=os.path.getmtime)
print('Run moi nhat:', run)
print('===== TEST metrics =====')
!find "{run}test_log" -type f -exec grep -hE "TEST results|recall|hit_ratio|ndcg" {{}} + 2>/dev/null
print('===== VALIDATION theo epoch (recall / iou) =====')
!find "{run}log" -type f -exec grep -hE "VALIDATION" {{}} + 2>/dev/null
print('===== Checkpoints da luu tren Drive =====')
!ls -lah "{run}ckpt/" 2>/dev/null

## Ghi chú
- Weight + log nằm sẵn ở **Drive / MyDrive / Colab Notebooks / DIVRS_output** — khỏi cần tải/copy thêm.
- Mỗi `epoch_N.pth` ~30MB → 30 epoch ≈ 1GB trên Drive. Muốn gọn, sau khi xong xoá bớt epoch giữa, giữ epoch có val recall cao nhất (xem mục VALIDATION).
- **Baseline**: đổi `--flagfile` → `config/ml10m_mf.cfg` / `ml10m_ips.cfg` / `ml10m_cause.cfg`.
- Paper DIVRS (MF, ML-10M): **Recall@20=0.1691, NDCG@20=0.1128** — mục tiêu đối chiếu.